In [ ]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import requests
import json

url = 'https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001'

### 1. BeautifulSoup 기반 정적 스크래핑 (YES24)
`requests`를 이용해 서버의 응답을 그대로 받아와 파싱하는 방식입니다. JS 처리가 필요 없는 정적 리스트 수집에 적합합니다.

In [ ]:
def scrape_yes24_with_bs4(target_url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    resp = requests.get(target_url, headers=headers)
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    books = []
    items = soup.select('#yesBestList > li')
    
    for i, item in enumerate(items):
        title_tag = item.select_one('a.gd_name')
        if not title_tag: continue
        
        title = title_tag.get_text(strip=True)
        link = "https://www.yes24.com" + title_tag.get('href', '')
        author = item.select_one('span.info_auth').get_text(strip=True) if item.select_one('span.info_auth') else "미상"
        publisher = item.select_one('span.info_pub').get_text(strip=True) if item.select_one('span.info_pub') else "미상"
        price_tag = item.select_one('span.yes_b')
        price = price_tag.get_text(strip=True) if price_tag else "0"
        
        books.append({'순위': i + 1, '제목': title, '저자': author, '출판사': publisher, '가격': price, '링크': link})
        
    return pd.DataFrame(books)

df_yes24 = scrape_yes24_with_bs4(url)
df_yes24.head(10)

--- 
### 2. 멜론 TOP 100 종합 스크래퍼 (BS4 + JSON API 병합)

멜론 차트는 '좋아요 수'가 정적 HTML에 포함되어 있지 않습니다. 
- **Step 1**: 곡 기본 정보와 `songId`를 BS4로 추출합니다.
- **Step 2**: 100개의 `songId`를 한 번에 멜론 API에 요청하여 좋아요 수를 가져옵니다.
- **Step 3**: 데이터를 병합하여 최종 리포트를 생성합니다.

In [30]:
def scrape_melon_top100():
    header = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}
    url = "https://www.melon.com/chart/index.htm"
    response = requests.get(url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    rows = soup.select('tr.lst50, tr.lst100')
    chart_data = []
    song_ids = []
    
    for row in rows:
        try:
            rank = row.select_one('span.rank').text
            title = row.select_one('div.ellipsis.rank01 a').text
            artists = ", ".join([a.text for a in row.select('div.ellipsis.rank02 span a')])
            album = row.select_one('div.ellipsis.rank03 a').text
            
            diff = "-"
            rank_wrap = row.select_one('span.rank_wrap')
            if rank_wrap:
                bullet = rank_wrap.select_one('span.bullet')
                val = rank_wrap.select_one('span.t_size').text if rank_wrap.select_one('span.t_size') else ""
                if bullet:
                    classes = bullet.get('class', [])
                    if 'rank_up' in classes: 
                        diff = f"▲{val}"
                    elif 'rank_down' in classes:
                        diff = f"▼{val}"
                    elif 'rank_new' in classes: 
                        diff = "NEW"
            
            song_id = row['data-song-no']
            song_ids.append(song_id)
            
            chart_data.append({'순위': int(rank), '변동': diff, '곡ID': song_id, '제목': title, '가수': artists, '앨범': album})
        except Exception:
            continue
    
    if song_ids:
        like_url = f"https://www.melon.com/commonlike/getSongLike.json?contsIds={','.join(song_ids)}"
        like_resp = requests.get(like_url, headers=header)
        likes_dict = {str(item['CONTSID']): item['SUMMCNT'] for item in like_resp.json()['contsLike']}
        for item in chart_data: 
            item['좋아요'] = likes_dict.get(item['곡ID'], 0)
    
    df = pd.DataFrame(chart_data)
    return df[['순위', '변동', '제목', '가수', '앨범', '좋아요']]

df_melon = scrape_melon_top100()
df_melon.head(10)

--- 
### 3. Billboard HOT 100 스크래핑
빌보드 차트는 클래스명이 길고 복잡하므로 텍스트와 계층 구조를 이용해 수집합니다.

In [32]:
def scrape_billboard_hot100():
    url = "https://www.billboard.com/charts/hot-100/"
    header = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}
    response = requests.get(url, headers=header)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    songs = []
    # 행 컨테이너 리스트 추출
    items = soup.find_all(class_='o-chart-results-list-row-container')
    
    for item in items:
        try:
            # 순위: 첫 번째 li 태그 안의 텍스트가 있는 span
            rank = item.find('span', class_='c-label').get_text(strip=True)
            
            # 제목: 행 안의 유일한 h3 태그
            title_elem = item.find('h3')
            if not title_elem:
                continue
            title = title_elem.get_text(strip=True)
            
            # 가수명: h3 태그 바로 다음으로 오는 span 요소
            artist_elem = title_elem.find_next_sibling('span')
            artist = artist_elem.get_text(strip=True) if artist_elem else "미상"
            
            songs.append({'순위': rank, '제목': title, '가수': artist})
        except Exception:
            continue
            
    return pd.DataFrame(songs)

df_billboard = scrape_billboard_hot100()
df_billboard.head(10)

---
### 4. 실전: 네이버 웹툰 하이브리드 스크래퍼 (Playwright + BS4)
동적인 '별점순' 정렬 클릭은 **Playwright**로 처리하고, 결과 HTML 파싱은 **BeautifulSoup**으로 처리하여 효율을 높입니다.

In [ ]:
from playwright.async_api import async_playwright
import asyncio

async def scrape_naver_webtoon_star_score():
    url = 'https://comic.naver.com/webtoon'
    # '별점순' 버튼 셀렉터
    btn_star_score = "#container > div.component_wrap.type2 > div.ComponentHead__component_head--O2tPr > div > div > button:nth-child(4)"
    container_selector = "div[class^='WeekdayMainView__daily_all_item']"

    async with async_playwright() as p:
        # 1. 브라우저 실행 및 페이지 이동
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url)
        
        # 2. '별점순' 버튼 클릭
        await page.wait_for_selector(btn_star_score)
        await page.click(btn_star_score)
        
        # 3. 데이터 갱신 대기 (네트워크 상태 고려 2초)
        await asyncio.sleep(2)
        
        # 4. 결과 HTML 소스 획득 후 즉시 종료
        html_content = await page.content()
        await browser.close()
        
    # 5. BeautifulSoup으로 정밀 파싱 수행
    soup = BeautifulSoup(html_content, 'html.parser')
    result = {}
    weekday_containers = soup.select(container_selector)
    
    for container in weekday_containers:
        # 요일 이름 추출
        day_name_tag = container.select_one('h3')
        if not day_name_tag: 
            continue
        day_name = day_name_tag.get_text(strip=True)
        
        # 뱃지(UP, 신작 등)를 제외하고 오직 제목 텍스트만 추출하기 위한 정밀 셀렉터
        # ContentTitle__title 클래스를 명시하여 순수 제목만 타겟팅
        title_elements = container.select('span[class*="ContentTitle__title"]')
        titles = []
        for el in title_elements:
            txt = el.get_text(strip=True)
            # 뱃지 정보와 섞이지 않도록 필터링 (선택 사항: 클래스명이 ContentBadge__로 시작하는 요소는 이미 배제됨)
            if txt and txt not in ["UP", "신작", "휴재"]: 
                titles.append(txt)
        
        if titles:
            result[day_name] = titles
        
    # 6. Pandas 데이터프레임 변환 및 정규화
    df = pd.DataFrame(dict([ (k, pd.Series(v)) for k,v in result.items() ]))
    return df

df_webtoon_star = await scrape_naver_webtoon_star_score()
df_webtoon_star.head(20)

,월요웹툰,화요웹툰,수요웹툰,목요웹툰,금요웹툰,토요웹툰,일요웹툰
0,똑 닮은 딸,엄마를 만나러 가는 길,미연,이상한 나라의 솔,1초,오늘만 사는 기사,논 투아
1,날 먹는 건 금지양!,백작가의 비밀스런 시녀님,별을 품은 소드마스터,"풍작이에요, 마왕님!",그냥 선생님,"다비, 아찔하게 흐르는",당신의 라이언
2,피폐물을 힐링물로 만드는 방법,가공낙원,마침표의 유예기간,소심한 호랭이 코코,개집사,탑아이돌의 막내 멤버가 되었다,마왕의 고백
3,대공비가 체질입니다,전리품 공작부인,용한소녀,도나츠와 서커스,사자의 서,스쿨 오브 스트릿,키스는 자기 전에
4,달로 만든 아이,청춘 러브썸,미래의 골동품 가게,이세계 캠핑으로 힐링 라이프,상남자,어글리후드,모럴리스 스캔들
5,악당 가족이 독립을 반대한다,사람의 탈,수인 보호소에서 남주를 입양해 버렸다,마리오네타,나 혼자 탑에서 농사,위탁가족!,악녀는 조용히 살고 싶을 뿐인데!
6,함부로 친절하지 말라,성검전설,"시바, 만만치 않다",게임 속 바바리안으로 살아남기,성북구 비둘기 이헌서,토끼 여주의 새엄마가 되었다,내 최애는 막차를 탄다
7,제국 최고의 악녀를 사랑하게 되었습니다,그냥 선생님,뽀대작렬,별정직 공무원,우리는 후라이족,아홉수 우리들,고대동물기
8,해피 페이스,아름다운 줄리엣을 위하여,그랜드 피날레,김오진과 이상한 동물들,광마회귀,용사파티만화,오리짱!
9,나의 보이소프렌드,연우의 순정,함부로 길들이지 마시오!,사내연애 사절!,합법해적 파르페,디펜스 게임의 폭군이 되었다,아카데미에 위장취업당했다
